# G-Eval with Claude as Judge

This notebook uses the **G-Eval** framework (via the `deepeval` library) with **Claude Opus** (`claude-opus-4-20250514`) as the judge model to evaluate LLM-generated popular-science summaries of arXiv articles.

**Why G-Eval?** G-Eval uses chain-of-thought reasoning to produce more calibrated evaluation scores. The framework auto-generates detailed evaluation steps from high-level criteria, then scores outputs accordingly.

**Evaluation criteria** (identical to `G_Eval_gpt.ipynb` for cross-judge comparison):
- Style transfer intensity
- Content preservation
- Naturalness and readability

**Scoring:** 1 (worst) to 5 (best), with a total rating.

**Domains:** CS, Physics, Math, Economics (50 articles each, 200 total).

**Mentee:** Dorothy / Xinyue Yuan

## 1. Setup & Installs

Install the required packages: `deepeval` for the G-Eval framework, `anthropic` for the Claude SDK, and `instructor` for structured-output support with Anthropic models.

In [ ]:
!pip install -q deepeval anthropic instructor

## 2. Imports & API Setup

Load the Anthropic API key from Colab Secrets and import all dependencies.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import os
import time
import anthropic
import instructor
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from google.colab import userdata

# Set API key for the Anthropic client
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

CLAUDE_MODEL = "claude-opus-4-20250514"
print(f"Using judge model: {CLAUDE_MODEL}")

## 3. Custom Claude LLM Wrapper for DeepEval

DeepEval's `GEval` metric expects an LLM that implements the `DeepEvalBaseLLM` interface. Since deepeval's pinned `anthropic` SDK version may not include `claude-opus-4-20250514`, we create a custom wrapper that uses the Anthropic SDK directly.

The wrapper uses the `instructor` library for structured output (JSON schema enforcement), which is how deepeval internally extracts scores from the judge model.

In [ ]:
class ClaudeOpusJudge(DeepEvalBaseLLM):
    """Custom DeepEval LLM wrapper for Claude Opus via the Anthropic SDK."""

    def __init__(self, model_name=CLAUDE_MODEL):
        self.model_name = model_name
        # Plain Anthropic client (for text generation)
        self._client = anthropic.Anthropic()
        # Instructor-patched client (for structured/JSON output)
        self._instructor_client = instructor.from_anthropic(self._client)

    def load_model(self):
        return self._client

    def generate(self, prompt: str, schema=None):
        """
        Generate a response from Claude.
        If a pydantic schema is provided, use instructor for structured output.
        Otherwise, return plain text.
        """
        if schema:
            # Structured output: instructor enforces the JSON schema
            return self._instructor_client.messages.create(
                model=self.model_name,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
                response_model=schema,
            )
        else:
            # Plain text generation
            response = self._client.messages.create(
                model=self.model_name,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.content[0].text

    async def a_generate(self, prompt: str, schema=None):
        """Async variant — falls back to sync for simplicity."""
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name


# Instantiate the judge model
claude_judge = ClaudeOpusJudge()
print(f"Claude judge wrapper ready: {claude_judge.get_model_name()}")

## 4. Helper Functions

These functions fetch arXiv article HTML, strip boilerplate (navigation UI, table of contents), and extract the first ~4,000 characters of article prose. Identical to `G_Eval_gpt.ipynb`.

In [ ]:
def fetch_html_body_content(html_url):
    """Fetch an arXiv HTML page and extract the body text."""
    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("Warning: HTML Fail:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # Truncate at Acknowledgment section
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]
    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    return text, "html_success"


# --- Boilerplate patterns to strip from arXiv HTML ---
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "x",
}

BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",
    "Content selection saved",
    "Describe the issue below",
]


def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble (nav UI + table of contents), then return
    the first max_chars of actual article prose.
    """
    lines = full_text.splitlines()

    # Phase 1: Find where the real article prose begins
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: From body_start onward, drop remaining boilerplate lines
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        if stripped in BOILERPLATE_EXACT:
            continue
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]


print("Helper functions loaded.")

## 5. Define the G-Eval Metric

We use the **exact same evaluation criteria** as `G_Eval_gpt.ipynb` — the only difference is the judge model (Claude Opus instead of GPT-5.2).

G-Eval will automatically generate chain-of-thought evaluation steps from this criteria, then use them to score each summary.

In [ ]:
# Define the G-Eval metric with Claude Opus as the judge
# Criteria text is identical to G_Eval_gpt.ipynb for cross-judge comparison
judge_metric = GEval(
    name="G-Eval-Judge",
    criteria=(
        """
Evaluate the quality of summaries written for a news article.
Rate each summary on four dimensions:
style transfer intensity, content preservation, naturalness and readability.
You should rate on a scale from 1 (worst) to 5 (best).

Provide your evaluation as follows:
Explanation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Explanation:' and 'Total rating:' in your answer.
"""
    ),
    model=claude_judge,  # Our custom Claude Opus wrapper
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT,
    ],
)

print("G-Eval metric defined with Claude Opus as judge.")

## 6. Data Loading & Evaluation Loop

We process all four domains. Each domain has its own CSV with 50 arXiv articles and LLM-generated summaries.

For each row, we:
1. Fetch the article HTML and extract a prose snippet
2. Create a `LLMTestCase` with the summary as `actual_output` and the article snippet as `retrieval_context`
3. Run G-Eval's `.measure()` method, which scores the summary using Claude Opus
4. Collect scores and explanations into the output DataFrame

> **Note:** The GPT version (`G_Eval_gpt.ipynb`) evaluates Llama summaries (`llama_prompt` column). We follow the same pattern here. Adjust the `summary_column` variable below if you want to evaluate a different model's summaries.

In [ ]:
# Domain configurations: (domain_label, input_csv, summary_column, output_csv)
DOMAINS = [
    ("cs",      "results_llama_cs.csv",   "llama_prompt", "results_llama_cs_geval_claude.csv"),
    ("physics", "results_llama_phy.csv",  "llama_prompt", "results_llama_phy_geval_claude.csv"),
    ("math",    "results_llama_math.csv", "llama_prompt", "results_llama_math_geval_claude.csv"),
    ("econ",    "results_llama_econ.csv", "llama_prompt", "results_llama_econ_geval_claude.csv"),
]

all_results = []  # Collect DataFrames for cross-domain analysis

for domain_label, input_csv, summary_column, output_csv in DOMAINS:
    print(f"\n{'='*60}")
    print(f"Processing domain: {domain_label.upper()}")
    print(f"{'='*60}")

    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"  File not found: {input_csv} — skipping this domain.")
        continue

    print(f"  Loaded {len(df)} rows from {input_csv}")
    scored_data = []

    for idx, row in df.iterrows():
        print(f"--- Processing Row {idx} ---")

        html_url = row.get("html_url")
        article, status = fetch_html_body_content(html_url)

        if status != "html_success":
            print(f"  Row {idx} failed: {status}")
            continue

        summary = row[summary_column]
        article_snippet = get_article_snippet(article)

        # Create the G-Eval test case
        test_case = LLMTestCase(
            input="Summary Evaluation Task",
            actual_output=summary,
            retrieval_context=[article_snippet],
        )

        try:
            # Run G-Eval measurement with Claude as judge
            judge_metric.measure(test_case)

            # Collect results
            row_result = row.to_dict()
            row_result["G-Eval rate"] = judge_metric.score
            row_result["G-Eval Explanation"] = judge_metric.reason
            scored_data.append(row_result)

            print(f"  Rate: {judge_metric.score}")
            print(f"  Explanation: {judge_metric.reason}\n")

        except Exception as e:
            print(f"  Row {idx} evaluation error: {e}")

    # Save per-domain results
    if scored_data:
        df_output = pd.DataFrame(scored_data)
        df_output.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"  Saved {len(df_output)} rows to {output_csv}")

        # Tag domain for aggregation
        df_output["domain"] = domain_label
        all_results.append(df_output)
    else:
        print(f"  No results generated for {domain_label}.")

print(f"\nDone! Processed {len(all_results)} domain(s).")

## 7. Results Aggregation

Combine all domain results and compute per-domain average G-Eval scores.

In [ ]:
# Combine all domains into one DataFrame
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    combined_df["G-Eval rate"] = pd.to_numeric(combined_df["G-Eval rate"], errors="coerce")

    # Per-domain summary statistics
    domain_stats = combined_df.groupby("domain")["G-Eval rate"].agg(
        count="count",
        mean="mean",
        std="std",
        median="median",
        min="min",
        max="max",
    ).round(2)

    print("Per-domain Claude G-Eval rating statistics:")
    print(domain_stats)
    print(f"\nOverall mean rating: {combined_df['G-Eval rate'].mean():.2f}")
    print(f"Total articles rated: {combined_df['G-Eval rate'].notna().sum()}")
else:
    print("No results to aggregate — check the evaluation loop above for errors.")

## 8. Visualization

### 8a. Average G-Eval rating by domain (bar chart)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if all_results:
    domain_means = combined_df.groupby("domain")["G-Eval rate"].mean()
    domain_order = ["cs", "physics", "math", "econ"]
    domain_labels = ["CS", "Physics", "Math", "Economics"]
    colors = ["#4C78A8", "#F58518", "#E45756", "#72B7B2"]

    means = [domain_means.get(d, 0) for d in domain_order]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(domain_labels, means, color=colors, edgecolor="black", linewidth=0.5)

    # Add value labels on bars
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

    ax.set_ylabel("Average G-Eval Rating (0-1)")
    ax.set_title("Claude G-Eval Judge: Average Summary Rating by Domain")
    ax.set_ylim(0, 1.1)
    overall_mean = combined_df["G-Eval rate"].mean()
    ax.axhline(y=overall_mean, color="gray", linestyle="--",
               label=f"Overall mean: {overall_mean:.2f}")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

### 8b. Rating distribution histogram

In [ ]:
if all_results:
    fig, ax = plt.subplots(figsize=(8, 5))

    domain_order = ["cs", "physics", "math", "econ"]
    domain_labels = ["CS", "Physics", "Math", "Economics"]
    colors = ["#4C78A8", "#F58518", "#E45756", "#72B7B2"]

    for domain, color, label in zip(domain_order, colors, domain_labels):
        subset = combined_df[combined_df["domain"] == domain]["G-Eval rate"].dropna()
        ax.hist(subset, bins=20, alpha=0.6, label=label, color=color,
                edgecolor="black", linewidth=0.5)

    ax.set_xlabel("G-Eval Rating")
    ax.set_ylabel("Count")
    ax.set_title("Claude G-Eval Judge: Rating Distribution by Domain")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

## 9. Save Combined Results

Save the full combined DataFrame for downstream cross-judge comparison (Claude G-Eval vs GPT G-Eval).

In [ ]:
if all_results:
    combined_df.to_csv("results_all_domains_geval_claude.csv", index=False, encoding="utf-8-sig")
    print(f"Saved combined results: {len(combined_df)} rows to results_all_domains_geval_claude.csv")
else:
    print("No combined results to save.")